In [116]:
from sklearn.metrics import mean_squared_error
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [117]:
# Load Seattle data
seattle_df = pd.read_csv("Datasets/Seattle_Rescue_Plan.csv")

# Load in Portland data
portland_df = pd.read_csv("Datasets/Portland_Rescue_Plan_Project_Information.csv")

# Rename columns for ease of use
seattle_df.rename(columns={'Program ID': 'program_id', 
                   'Dept (Full Name)': 'dept_full',
                   'Dept (Acronym)': 'dept_acr',
                   'Seattle Rescue Plan': 'seattle_rescue_plan',
                   'Category of Investment': 'category',
                   'Item Name for the Public': 'public_name',
                   'Funding Source': 'funding_source',
                   'Item Main Objective': 'main_objective',
                   'Budgeted': 'budgeted',
                   'Expenditures': 'expenditures',
                   'Encumbrances': 'encumbrances',
                   'Total Spent  \n(Expenditures + Encumbrances)': 'total_spent',
                   'Program Status': 'program_status'
                   }, 
                   inplace=True)

# Check Seattle columns got renamed properly 
print(seattle_df.columns)



Index(['program_id', 'dept_full', 'dept_acr', 'seattle_rescue_plan',
       'category', 'public_name', 'funding_source', 'main_objective',
       'budgeted', 'expenditures', 'encumbrances', 'total_spent',
       'program_status'],
      dtype='object')


In [118]:
print(seattle_df['program_id'].shape)

(86,)


In [119]:
# Split Seattle 2021 DFs by year 
print(seattle_df["seattle_rescue_plan"].value_counts())

# Most programs fall under SRP1 

# Method for finding with program IDs correspond to a particular SRP (1 - 4). 
# Parameter: the SRP plan number you're lookign for (ex: 'SRP1', 'SRP2'...)
def ids_by_year(year_string):
    srp_ids = seattle_df.loc[seattle_df['seattle_rescue_plan'] == year_string, 'program_id']
    sorted_ids = srp_ids.sort_values()
    srp_id_list = sorted_ids.tolist()
    return(srp_id_list)


srp1_ids = ids_by_year('SRP1')
srp2_ids = ids_by_year('SRP2')
srp3_ids = ids_by_year('SRP3')
srp4_ids = ids_by_year('SRP4')  

print(srp1_ids)
print(srp2_ids)
print(srp3_ids)
print(srp4_ids)


seattle_rescue_plan
SRP1    56
SRP3    22
SRP2     6
SRP4     2
Name: count, dtype: int64
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 92, 102]
[93, 94, 95, 96, 97, 98]
[56, 57, 58, 59, 61, 62, 63, 64, 65, 66, 67, 68, 70, 71, 77, 78, 79, 80, 81, 82, 83, 84]
[100, 101]


#### When searching for a program in the performance report, *FIND IT IN THE MOST RECENT SUMMARY POSSIBLE*

Some program IDs are missing -- they correspond to new rounds of funding added to projects established in previous rescue plans \
EX: program_id 12 and program_87 are both the "Seattle Promise" program in 2021 and 2023 respectively \
Only the first appearance of the program will be in the DF, but it will appear in the latest summary report that it received funding during


In [120]:
# Sorted df for identify names and IDs

sorted_seattle = seattle_df.sort_values(by = 'program_id')
sorted_seattle = sorted_seattle[['program_id', 'public_name', 'budgeted', 'total_spent', 'seattle_rescue_plan']]

print(sorted_seattle.head(10))

    program_id                                        public_name  budgeted  \
10           1                           Summer Campus Activation    140500   
7            2  Seattle Rescue Plan Performance Monitoring & E...   1105000   
8            3                    Affordable Seattle and CiviForm    920995   
78           4                                 Library Open Hours    465317   
39           5     CiviForm (Product Development and Maintenance)   1657632   
40           6                                     Digital Equity    500000   
41           7                 Telework Capability for City Staff   2700000   
42           8                              Restore City Staffing   6260000   
43           9                      City of Seattle Cybersecurity   1500000   
16          10  Support COVID-19 Mitigation and Prevention in ...    183628   

    total_spent seattle_rescue_plan  
10       140500                SRP1  
7        563400                SRP1  
8        427985 

In [121]:
# Columns to add for performance evaluation
# [benef_count, benef_type, equity_impact_%, performance_score]

# Method: find the public_name of the program with the ID of interest
# Check for that program name in ALL of the summary reports -- pull from the most recent one
# Add columns as necessary


# Make a copy of seattle data for mapping performance outcomes
seattle_eval_df = sorted_seattle

# Method for returning the name of the program of interest
def name_finder(id):
    print(seattle_eval_df.loc[seattle_eval_df['program_id'] == id, 'public_name'])

# Method for adding benef_count and benef_type column values for a given program by ID #
def add_benef_columns(program_id, type, count):
    seattle_eval_df.loc[seattle_eval_df['program_id'] == program_id, 'benef_type'] = type
    seattle_eval_df.loc[seattle_eval_df['program_id'] == program_id, 'benef_count'] = count


In [122]:
# Adding benef_type and benef_count for Program #1

# name_finder(1) # Summer Campus Activation # DONE
add_benef_columns(1, 'attendees', 30000) 

# Test it actually works:
# print(seattle_eval_df[seattle_eval_df['program_id'] == 1])


# name_finder(2) # Seattle Rescue Plan Performance Monitoring & E... 
#!! Missing impact info

# name_finder(3) # Affordable Seattle and CiviForm OR CiviForm (Affordable Seattle Program Management)
add_benef_columns(3, 'residents', 16255)
#!! CONSULT TEAM AND COME BACK TO THIS LATER -- should benef_type be applicants? residents?

# name_finder(4) # Library Open Hours
add_benef_columns(4, 'employees', 34)
#!! CONSULT TEAM -- employees or libraries?

# name_finder(5) # CiviForm (Product Development and Maintenance)
#!! Missing impact info

# name_finder(6) # Digital Equity # DONE
add_benef_columns(6, 'residents', 2200)

# name_finder(7) # Telework Capability for City Staff
#!! Missing impact info -- impact numbers not reported (intentionallty)

# name_finder(8) # Restore City Staffing
#!! CONSULT TEAM -- Program impact goes beyond number of employees rehired

# name_finder(9) # City of Seattle Cybersecurity
#!! Missing impact info

# name_finder(10) # Support COVID-19 Mitigation and Prevention in...
#!! Missing impact info

# name_finder(11) # Priority Hire
add_benef_columns(11, 'employees', 9)
#!! Consult with team, not sure if it should be "residents" or "employees"

# name_finder(12) # Seattle Promise 
add_benef_columns(12, 'residents', 1054)
#!! Need to check-in with team to confirm

# name_finder(13) # Childcare Wage Allotment
add_benef_columns(13, 'employees', 3500)
#!! Pretty sure it's done, need to confirm focus on staff vs children

# name_finder(14) # Enhanced Parks Maintenance # DONE
add_benef_columns(14, 'employees', 22)

# name_finder(15) # Community Activation of Public Parks
add_benef_columns(15, 'residents', 78755)
#!! Confirm what benef type to keep track of with team

# name_finder(16) # Community Programming: Wading Pools and Rec N' ...
add_benef_columns(16, 'residents', 52800)
#!! Confirm benef count with team -- residents? attendees?

# name_finder(17) # Cultural Districts Recovery Grants
add_benef_columns(17, 'employees', 335)
#!! Confirm benef type 

# name_finder(18) # Cultural Organization Reopening Grants
add_benef_columns(18, 'employees', 4507)
#!! Confirm benef type

# name_finder(19) # Technical Assistance for Rehiring Artists and Cultural Workers
add_benef_columns(19, 'employees', 32)
#!! Confirm benef type and count

# name_finder(20) # Hope Corps (Beloved Campaign)
add_benef_columns(20, 'employees', 89)
#!! Confirm benef type and count with team 

# name_finder(21) # Created Commons
add_benef_columns(21, 'attendees', 22775)
#!! Confirm benef type and count

# name_finder(22) # Maritime Apprenticeships for Youth and Young 
#!! Missing impact information 

# name_finder(23) # Digital Bridge # DONE
add_benef_columns(23, 'residents', 276)

# name_finder(24) # Small Business Stabilization Fund
add_benef_columns(24, 'businesses', 285)
#!! Confirm benef type with team

# name_finder(25) # Small Business Recovery Fund
add_benef_columns(25, 'businesses', 264)

# name_finder(26) # Small Business Digital Access || Technology Access
add_benef_columns(26, 'businesses', 118)

# name_finder(27) # Small Business Financial Health and Accounting || Small Business Accounting and Business Consulting # DONE
add_benef_columns(27, 'businesses', 671)

# name_finder(28) # Small Business Recovery Navigation
add_benef_columns(28, 'businesses', 1209)
#!! Confirm benef count with team 

# name_finder(29) # Shop to the Beat (Small Business and Creatives
add_benef_columns(29, 'businesses', 49)
#!! Missing some information, change benef type?

# name_finder(30) # Small Business Legal Technical Assistance
add_benef_columns(30, 'businesses', 321)
#!! Confirm benef_count with team 

# name_finder(31) # Seattle Restored (Empty Storefronts) # DONE
add_benef_columns(31, 'businesses', 31)

# name_finder(32) # Tenant Improvement Fund (Commercial Afford
add_benef_columns(32, 'businesses', 63)
#!! Confirm benef_count with team 

# name_finder(33) # Neighborhood Economic Recovery 
add_benef_columns(33, 'businesses', 2727)
#!! Confirm beneft_type and benef_count with team 

# name_finder(34) # Missing in dataset
# Listed as "downtown activation"

# name_finder(35) # Downtown Workforce Development
add_benef_columns(35, 'employees', 309)
#!! Confirm benef_type and benef_count with team 

# name_finder(36) # Enhanced Diaper Distribution
add_benef_columns(36, 'diapers', 670000)
#!! Consult with team, benef_type here is very unusual

# name_finder(37) #  Good Food Kitchens 
add_benef_columns(37, 'meals served', 55288)
#!! Consult with team, reports are a bit confusing + odd benef_type

# name_finder(38) # Childcare Facilities
add_benef_columns(38, 'residents', 599)
#!! Consult with team about benef_type

# name_finder(39) # Behavioral Health For Youth and Families
add_benef_columns(39, 'residents', 2157)
#!! Consult team about benef_type and benef_count

# name_finder(40) # Gender-Based Violence Response Services
add_benef_columns(40, 'residents', 638)
#!! Consult team about benef_type and benef_count

# name_finder(41) # Capacity Building for Homeless Service Providers  || Capacity Building for Housing Providers ????
add_benef_columns(41, 'households', 4021)
#!! Consult team about benef_type and benef_count 

# name_finder(42) # Expanded Homelessness Diversion # DONE
add_benef_columns(42, 'residents', 3000)

# name_finder(43) #  Rapid Rehousing # DONE
#!! Grouped in with program # 42, DO NOT FILL OUT

# name_finder(44) # Enhanced Shelter and Outreach
#!! Missing information

# name_finder(45) #  Safe Lots (RV/ Vehicles)
#!! Missing information 

# name_finder(46) # Tiny Home Villages
#!! Missing information

# name_finder(47) # Multifamily Housing Acquisition Capital
add_benef_columns(47, 'households', 160)
#!! Not sure about benef_type: households? units? 

# name_finder(48) # Capacity Building for Housing Providers
#!! This is logged as program # 41 in the dataset for some reson? It's #48 in reports though

# name_finder(49) #  Stay Healthy Streets (Neighborhood Greenways)
add_benef_columns(49, 'miles', 20)
#!! Odd benef_type

# name_finder(50) # Safe Start Business Recovery Program || Safe Starts # DONE
add_benef_columns(50, 'business', 245)
#!! benef_type is ambiguous

# name_finder(51) # Creative Industries Small Business Technical A... 
#!! Missing benef_type and benef_count

# name_finder(52) # Downtown Activation: Welcome Back Weeks
#!! Missing benef_type and benef_count

# name_finder(53) # Maintenance Expenses for City Owned Buildings
#!! Missing info, lumped in with other programs

# name_finder(54) # Scholarships for Childcare 
add_benef_columns(54, 'residents', 609)
#!! benef_type not clear

# name_finder(55) # Seattle Relief Fund # DONE
add_benef_columns(55, 'residents', 26200)

# name_finder(56) # Federal Funds Project Management Staffing (CBO)
#!! Missing benef_type and benef_count

# name_finder(57) # Federal Funds Project Management Staffing (FAS)
#!! Missing benef_type and benef_count

# name_finder(58) # COVID Mitigation in Shelters
#!! Missing benef_type and benef_count

# name_finder(59) #  Maintain Enhanced Shelter Units (SODO and Keiro)
#!! Missing info, lumped in with program # 42

# name_finder(60) # N/A -- missing from dataset

# name_finder(61) # Food Assistance
add_benef_columns(61, 'meals', 306815)
#!! Not sure about ben_type and ben_count

# name_finder(62) # City Employee COVID Vaccine Verification System
#!! Missing benef_type and benef_count

# name_finder(63) # Federal Funds Project Management Staffing (OEM)
#!! Missing benef_type and benef_count

# name_finder(64) # Pilot Prescription Food Program
#!! Missing benef_type and benef_count 

# name_finder(65) # Return to Office and Future of Work
#!! Missing benef_type and benef_count

# name_finder(66) # Clean City Initiative Expansion (SDOT)
add_benef_columns(66, 'lbs of trash', 528820)
#!! Not sure about benef_type and benef_count 

# name_finder(67) # Clean City Initiative Expansion (SPR)
#!! Missing information, lumped in with program # 66

# name_finder(68) # Clean City Initiative Expansion (SPU)
#!! Missing information, lumped in with program # 66

# name_finder(69) # NA -- missing from dataset

# name_finder(70) # Cultural Organization Funding
#!! Missing benef_type and benef_count 

# name_finder(71) # Low-Acuity Response Implementation Plan
#!! Missing benef_type and benef_count

# name_finder(72) # N/A -- missing from dataset 

# name_finder(73) # N/A -- missing from dataset

# name_finder(74) # N/A -- missing from dataset

# name_finder(75) # N/A -- missing from dataset

# name_finder(76) # N/A -- missing from dataset

# name_finder(77) # Support for American Indian and Alaskan Native Populations
#!! Missing benef_type and benef_count

# name_finder(78) # Mobile Mental and Behavioral Health Crisis Services
#!! Missing benef_type and benef_count

# name_finder(79) #  Support for Survivors of Gender-Based Violence
#!! Missing info, lumped in with program # 40

# name_finder(80) # Regional Peacekeepers Collective
#!! Missing benef_type and benef_count

# name_finder(81) # Seattle City Council Staffing
#!! Missing benef_type and benef_count

# name_finder(82) # Seattle Fire Department Payroll Expenses
#!! Missing benef_type and benef_count 

# name_finder(83) # Seattle Public Library Vandalism Repair
#!! Missing benef_type and benef_count 

# name_finder(84) # Fresh Bucks # DONE
add_benef_columns(84, 'households', 12000)

# name_finder(85) # N/A -- missing from dataset

# name_finder(86) # N/A -- missing from dataset

# name_finder(87) # N/A -- missing from dataset

# name_finder(88) # N/A -- missing from dataset

# name_finder(89) # N/A -- missing from dataset

# name_finder(90) # N/A -- missing from dataset

# name_finder(91) # N/A -- missing from dataset

# name_finder(92) # Affordable Housing Capital ||  Non-PSH Housing Projects
#!! Lumped in with program # 41

# name_finder(93) # Senior Services
#!! Missing ben_type and ben_count

# name_finder(94) # Rental Assistance and Eviction Prevention || Household Assistance: Eviction Prevention
#!! Missing ben_type and ben_count

# name_finder(95) # Madison Street Bus Rapid Transit (BRT)
#!! Missing ben_type and ben_count

# name_finder(96) # Streetcar Operations and Maintenance
#!! Missing ben_type and ben_count

# name_finder(97) # Monorail Operations and Maintenance
#!! Missing ben_type and ben_count

# name_finder(98) #  Operating Grant for McCaw Hall
add_benef_columns(98, 'venue', 1)
#!! Not sure about ben_type

# name_finder(99) # N/A -- missing from dataset

# name_finder(100) # Low Income Home Energy Assistance Program
#!! Not sure if I should include, ben_type = 1 (it's a grant for another program)

# name_finder(101) # Hope Corps (Beloved Campaign)
#!! Bundled into program # 20

# name_finder(102) # Storefront Repair Fund # DONE
add_benef_columns(102, 'businesses', 480)
#!! Ongoing